# Validate and clean duplicate and invalid readings

This notebook starts from `cleaned_dataset.csv`, checks station-hour identities and value validity, and writes any changes to a **new** `cleaned_dataset_validated.csv` file. It does not modify the input. Missing values left by the earlier cleaning step are reported, not filled again.

The rules below only automatically invalidate clearly impossible values (negative concentrations, rainfall, or wind speed; non-positive pressure; unparseable numeric readings; invalid timestamps/stations; and unknown wind-direction labels). Unusually large positive measurements are reported for inspection rather than discarded without a defensible domain threshold. Conflicting records for the same station and hour stop the cleaning step for manual review.

In [1]:
from pathlib import Path
import pandas as pd

candidate_dirs = [Path.cwd(), Path.cwd() / "Notebooks"]
DATA_DIR = next(
    (directory for directory in candidate_dirs
     if (directory / "cleaned_dataset.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find cleaned_dataset.csv in the current directory or its Notebooks subdirectory."
    )

SOURCE_FILE = DATA_DIR / "cleaned_dataset.csv"
OUTPUT_FILE = DATA_DIR / "cleaned_dataset_validated.csv"

df_original = pd.read_csv(SOURCE_FILE)
print(f"Loaded {len(df_original):,} rows from {SOURCE_FILE}")
print(f"Columns: {len(df_original.columns)}")

Loaded 418,753 rows from C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\Notebooks\cleaned_dataset.csv
Columns: 19


## Validate record identity

A reading is identified by station and hour, not by whether every column happens to match. Invalid or incomplete station/timestamp identities cannot be reliably deduplicated, so those rows are excluded from the new output and counted. Repeated station-hour records with identical readings can be collapsed safely; if readings conflict, the notebook stops rather than choosing one arbitrarily.

In [2]:
EXPECTED_STATIONS = {
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong",
}
TIME_COLUMNS = ["year", "month", "day", "hour"]
READING_COLUMNS = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM",
]
REQUIRED_COLUMNS = ["station", *TIME_COLUMNS, *READING_COLUMNS]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df_original.columns))
if missing_columns:
    raise KeyError(f"Required columns missing from the dataset: {missing_columns}")

time_parts = df_original[TIME_COLUMNS].apply(pd.to_numeric, errors="coerce")
integer_time_parts = time_parts.notna() & time_parts.mod(1).eq(0)
safe_time_parts = time_parts.fillna(-1).astype("int64")
timestamps = pd.to_datetime(
    {
        "year": safe_time_parts["year"],
        "month": safe_time_parts["month"],
        "day": safe_time_parts["day"],
        "hour": safe_time_parts["hour"],
    },
    errors="coerce",
)
valid_timestamp = timestamps.notna() & integer_time_parts.all(axis=1)
station_key = df_original["station"].astype("string").str.strip().str.casefold()
valid_station = station_key.isin(EXPECTED_STATIONS)
invalid_identity = ~(valid_timestamp & valid_station)

print(f"Rows with invalid/missing station or timestamp identity: {int(invalid_identity.sum()):,}")
print(f"  Invalid timestamps: {int((~valid_timestamp).sum()):,}")
print(f"  Missing or unexpected stations: {int((~valid_station).sum()):,}")

work = df_original.loc[~invalid_identity].copy()
work["_timestamp"] = timestamps.loc[~invalid_identity]
work["_station_key"] = station_key.loc[~invalid_identity]
KEY_COLUMNS = ["_station_key", "_timestamp"]

duplicate_key_mask = work.duplicated(KEY_COLUMNS, keep=False)
duplicate_key_rows = work.loc[duplicate_key_mask]
if duplicate_key_rows.empty:
    conflicting_keys = pd.Series(dtype="int64")
else:
    distinct_reading_counts = duplicate_key_rows.groupby(
        KEY_COLUMNS, dropna=False
    )[READING_COLUMNS].apply(lambda rows: len(rows.drop_duplicates()))
    conflicting_keys = distinct_reading_counts[distinct_reading_counts > 1]

print(f"Rows involved in repeated station-hour keys: {len(duplicate_key_rows):,}")
print(f"Conflicting station-hour keys needing manual review: {len(conflicting_keys):,}")
if not conflicting_keys.empty:
    display(conflicting_keys.to_frame("distinct_readings"))

Rows with invalid/missing station or timestamp identity: 0
  Invalid timestamps: 0
  Missing or unexpected stations: 0
Rows involved in repeated station-hour keys: 0
Conflicting station-hour keys needing manual review: 0


## Clean safe duplicates and invalid readings

Concentrations, rainfall, and wind speed cannot be negative; pressure must be positive. Non-numeric values in numeric columns are invalid. Wind direction is checked against the 16 standard compass points. Invalid individual readings become missing values so other useful measurements in that hour are retained. This notebook does not impose arbitrary maximum thresholds on positive concentrations.

In [3]:
if not conflicting_keys.empty:
    raise RuntimeError(
        "Conflicting readings share a station-hour key. Review the displayed groups and resolve them "
        "before writing cleaned_dataset_validated.csv. The source and backup remain unchanged."
    )

rows_before_deduplication = len(work)
work = work.drop_duplicates(
    subset=[*KEY_COLUMNS, *READING_COLUMNS],
    keep="first",
).copy()
duplicate_rows_removed = rows_before_deduplication - len(work)

NONNEGATIVE_COLUMNS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "RAIN", "WSPM"]
NUMERIC_COLUMNS = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
]
invalid_value_counts = {}
invalid_reading_masks = {}
for column in NUMERIC_COLUMNS:
    raw_values = work[column]
    numeric_values = pd.to_numeric(raw_values, errors="coerce")
    invalid = raw_values.notna() & numeric_values.isna()
    if column in NONNEGATIVE_COLUMNS:
        invalid |= numeric_values.lt(0)
    if column == "PRES":
        invalid |= numeric_values.le(0)
    invalid_value_counts[column] = int(invalid.sum())
    invalid_reading_masks[column] = invalid
    work[column] = numeric_values.mask(invalid)

COMPASS_DIRECTIONS = {
    "N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
    "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW",
}
wind_direction = work["wd"].astype("string").str.strip().str.upper()
invalid_wind_direction = wind_direction.notna() & ~wind_direction.isin(COMPASS_DIRECTIONS)
invalid_value_counts["wd"] = int(invalid_wind_direction.sum())
invalid_reading_masks["wd"] = invalid_wind_direction
work["wd"] = wind_direction.mask(invalid_wind_direction)

rows_with_invalid_readings = int(
    pd.DataFrame(invalid_reading_masks).any(axis=1).sum()
)

result = work.drop(columns=["_timestamp", "_station_key"])
result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

summary = pd.DataFrame(
    {
        "Check": ["Input rows", "Invalid identities", "Duplicate rows removed", "Rows with invalid readings", "Output rows"],
        "Count": [
            len(df_original),
            int(invalid_identity.sum()),
            duplicate_rows_removed,
            rows_with_invalid_readings,
            len(result),
        ],
    }
)
print("Cleaning summary")
display(summary.style.hide(axis="index").format({"Count": "{:,}"}))

invalid_readings_table = pd.DataFrame(
    {"Reading": list(invalid_value_counts), "Invalid cells": list(invalid_value_counts.values())}
)
print("Invalid reading cells by column")
display(
    invalid_readings_table.style
    .hide(axis="index")
    .format({"Invalid cells": "{:,}"})
)

remaining_nulls = (
    result[READING_COLUMNS].isna().sum().sort_values(ascending=False)
)
remaining_nulls = remaining_nulls[remaining_nulls > 0].rename("Remaining nulls")
remaining_nulls_table = remaining_nulls.rename_axis("Reading").reset_index()
print("Remaining missing readings")
if remaining_nulls_table.empty:
    print("None")
else:
    display(
        remaining_nulls_table.style
        .hide(axis="index")
        .format({"Remaining nulls": "{:,}"})
    )

print(f"\nSaved separate output: {OUTPUT_FILE}")
print(f"The source file was not modified. Duplicate rows removed: {duplicate_rows_removed:,}.")

Cleaning summary
Invalid reading cells by column
Remaining missing readings

Saved separate output: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\Notebooks\cleaned_dataset_validated.csv
The source file was not modified. Duplicate rows removed: 0.


Check,Count
Input rows,"418,753"
Invalid identities,0
Duplicate rows removed,0
Rows with invalid readings,0
Output rows,"418,753"


Reading,Invalid cells
PM2.5,0
PM10,0
SO2,0
NO2,0
CO,0
O3,0
TEMP,0
PRES,0
DEWP,0
RAIN,0


Reading,Remaining nulls
CO,"9,996"
NO2,"3,533"
O3,"3,436"
SO2,"1,901"
PM2.5,"1,359"
PM10,299
wd,8
TEMP,1
PRES,1
DEWP,1
